# NER Fine-tuning: NlpHUST/ner-vietnamese-electra-base

Nhận diện các aspect spans (BATTERY, CAMERA, PERFORMANCE...) trên **ViSD4SA** tiếng Việt.

**Đánh giá:** `seqeval` — span-level F1 (entity-level, chuẩn NER).

In [1]:
!pip install -q jsonlines transformers datasets seqeval accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 1. Cấu hình

In [2]:
from os import path

# ── Đổi đường dẫn tại đây nếu dùng Colab ──────────────────────────────────
ROOT = "/content"
# ──────────────────────────────────────────────────────────────────────────

DATASET    = path.join(ROOT, "dataset")
MODEL_SAVE = path.join(ROOT, "models/NER")
BASE_MODEL = "NlpHUST/ner-vietnamese-electra-base"

EPOCHS       = 10
LR           = 2e-5
BATCH_SIZE   = 32
MAX_SEQ_LEN  = 256
WARMUP_STEPS = 200
WEIGHT_DECAY = 0.01

## 2. Tải dữ liệu

Format ViSD4SA: `{"text": "...", "labels": [[start, end, "ASPECT#SENTIMENT"], ...]}`

In [3]:
import jsonlines

def load_jsonl(fp):
    with jsonlines.open(fp) as f:
        return list(f.iter())

train_raw = load_jsonl(path.join(DATASET, "train.jsonl"))
dev_raw   = load_jsonl(path.join(DATASET, "dev.jsonl"))
test_raw  = load_jsonl(path.join(DATASET, "test.jsonl"))

print(f"Train: {len(train_raw)} | Dev: {len(dev_raw)} | Test: {len(test_raw)}")
print("\nVí dụ:")
print("text  :", train_raw[0]["text"])
print("labels:", train_raw[0]["labels"])

Train: 7785 | Dev: 1112 | Test: 2225

Ví dụ:
text  : Pin Sài tầm 50h cho pin 100/100. Camera ổn ... tất cả đều OK ... nhân viên thế giới di động trần văn thời cà mau nhiệt tình và vui vẻ ...chúc các ae sức khỏe tốt và phục ok hoài nha....
labels: [[0, 31, 'BATTERY#POSITIVE'], [33, 42, 'CAMERA#POSITIVE'], [47, 60, 'GENERAL#POSITIVE'], [65, 181, 'SER&ACC#POSITIVE']]


## 3. Tiền xử lý

Chuyển span-level labels → BIO tags ở word level.

In [4]:
import unicodedata
import re

def clean_text(text):
    chars = [c if (unicodedata.category(c)[0] in ('L', 'N') or c == ' ') else ' ' for c in text]
    return re.sub(r' +', ' ', ''.join(chars)).strip().lower()


def build_bio_tags(text, labels):
    words = text.split()
    tags  = ["O"] * len(words)

    for start_char, end_char, label in labels:
        aspect     = label.split("#")[0]
        word_start = len(text[:start_char].split())
        word_end   = word_start + len(text[start_char:end_char].split())
        for i in range(word_start, min(word_end, len(tags))):
            tags[i] = ("B-" if i == word_start else "I-") + aspect

    clean_words, clean_tags = [], []
    for word, tag in zip(words, tags):
        w = clean_text(word)
        if w:
            clean_words.append(w)
            clean_tags.append(tag)

    return clean_words, clean_tags


def process_split(raw):
    result = []
    for item in raw:
        words, tags = build_bio_tags(item["text"], item["labels"])
        if words:
            result.append({"words": words, "tags": tags})
    return result


train_data = process_split(train_raw) + process_split(dev_raw)
test_data  = process_split(test_raw)

print(f"Train+Dev: {len(train_data)} | Test: {len(test_data)}")
print("\nVí dụ:")
print("words:", train_data[0]["words"][:8])
print("tags :", train_data[0]["tags"][:8])

Train+Dev: 8897 | Test: 2225

Ví dụ:
words: ['pin', 'sài', 'tầm', '50h', 'cho', 'pin', '100 100', 'camera']
tags : ['B-BATTERY', 'I-BATTERY', 'I-BATTERY', 'I-BATTERY', 'I-BATTERY', 'I-BATTERY', 'I-BATTERY', 'B-CAMERA']


## 4. Tokenization + Label Alignment

Căn chỉnh BIO labels với subword tokenization:
- Token đặc biệt `[CLS]`/`[SEP]` → `-100`
- Subword không phải đầu từ → `-100` (bỏ qua khi tính loss)

In [5]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer
import numpy as np
import torch

all_tags = sorted({t for d in train_data + test_data for t in d["tags"]})
label2id = {t: i for i, t in enumerate(all_tags)}
id2label = {i: t for i, t in enumerate(all_tags)}

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print(f"Số nhãn: {len(all_tags)}")
print(f"Nhãn:    {all_tags}")


def tokenize_and_align(words, tags):
    enc = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
    )
    labels, prev_word_id = [], None
    for word_id in enc.word_ids():
        if word_id is None:
            labels.append(-100)
        elif word_id != prev_word_id:
            labels.append(label2id[tags[word_id]])
        else:
            labels.append(-100)
        prev_word_id = word_id
    enc["labels"] = labels
    return enc


class NERDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        d = self.data[idx]
        return {k: torch.tensor(v) for k, v in tokenize_and_align(d["words"], d["tags"]).items()}


train_ds = NERDataset(train_data)
test_ds  = NERDataset(test_data)
print(f"\nTrain: {len(train_ds)} | Test: {len(test_ds)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Số nhãn: 21
Nhãn:    ['B-BATTERY', 'B-CAMERA', 'B-DESIGN', 'B-FEATURES', 'B-GENERAL', 'B-PERFORMANCE', 'B-PRICE', 'B-SCREEN', 'B-SER&ACC', 'B-STORAGE', 'I-BATTERY', 'I-CAMERA', 'I-DESIGN', 'I-FEATURES', 'I-GENERAL', 'I-PERFORMANCE', 'I-PRICE', 'I-SCREEN', 'I-SER&ACC', 'I-STORAGE', 'O']

Train: 8897 | Test: 2225


## 5. Fine-tune Model

In [6]:
import logging
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)

logging.getLogger("transformers").setLevel(logging.WARNING)


def compute_metrics(p):
    preds  = np.argmax(p.predictions, axis=2)
    labels = p.label_ids
    true_flat = [id2label[l].split("-")[-1] for p_row, l_row in zip(preds, labels)
                 for _, l in zip(p_row, l_row) if l != -100]
    pred_flat = [id2label[p].split("-")[-1] for p_row, l_row in zip(preds, labels)
                 for p, l in zip(p_row, l_row) if l != -100]
    entity_labels = [t for t in set(true_flat) if t != "O"]
    return {
        "f1":        f1_score(true_flat, pred_flat, labels=entity_labels, average="micro", zero_division=0),
        "precision": precision_score(true_flat, pred_flat, labels=entity_labels, average="micro", zero_division=0),
        "recall":    recall_score(true_flat, pred_flat, labels=entity_labels, average="micro", zero_division=0),
    }


model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(all_tags),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir=MODEL_SAVE,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors:   0%|          | 0.00/532M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ElectraForTokenClassification LOAD REPORT from: NlpHUST/ner-vietnamese-electra-base
Key                             | Status     |                                                                                      
--------------------------------+------------+--------------------------------------------------------------------------------------
electra.embeddings.position_ids | UNEXPECTED |                                                                                      
classifier.bias                 | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([9]) vs model:torch.Size([21])          
classifier.weight               | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([9, 768]) vs model:torch.Size([21, 768])

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,1.651173,1.340140,0.625030,0.628154,0.621938
2,1.062260,0.960359,0.728817,0.733574,0.724121
3,0.884802,0.882693,0.757440,0.747031,0.768143
4,0.741973,0.905443,0.758623,0.731779,0.787511
5,0.679705,0.854823,0.766671,0.763017,0.770362
6,0.607426,0.872996,0.768746,0.748674,0.789922
7,0.555935,0.881870,0.771289,0.759491,0.783460
8,0.513989,0.880116,0.771084,0.764886,0.777383
9,0.490406,0.912530,0.769445,0.750909,0.788919
10,0.460022,0.901905,0.771875,0.761832,0.782187


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

TrainOutput(global_step=2790, training_loss=0.8256184191686705, metrics={'train_runtime': 827.1917, 'train_samples_per_second': 107.557, 'train_steps_per_second': 3.373, 'total_flos': 4878506792066778.0, 'train_loss': 0.8256184191686705, 'epoch': 10.0})

## 6. Đánh giá (token-level)

In [7]:
from sklearn.metrics import classification_report
import numpy as np

preds_output = trainer.predict(test_ds)
preds  = np.argmax(preds_output.predictions, axis=2)
labels = preds_output.label_ids

# Bỏ token đặc biệt (-100), flatten, bỏ prefix B-/I-
true_flat = [id2label[l].split("-")[-1] for p_row, l_row in zip(preds, labels)
             for _, l in zip(p_row, l_row) if l != -100]
pred_flat = [id2label[p].split("-")[-1] for p_row, l_row in zip(preds, labels)
             for p, l in zip(p_row, l_row) if l != -100]

label_names = sorted({t for t in true_flat if t != "O"})
print(classification_report(true_flat, pred_flat, labels=label_names, digits=4))

              precision    recall  f1-score   support

     BATTERY     0.8367    0.8551    0.8458      8399
      CAMERA     0.8016    0.8622    0.8308      3904
      DESIGN     0.7610    0.8065    0.7831      2491
    FEATURES     0.7349    0.7550    0.7448      7838
     GENERAL     0.7374    0.7646    0.7508      8620
 PERFORMANCE     0.7667    0.7839    0.7752     11771
       PRICE     0.5612    0.5901    0.5753      1298
      SCREEN     0.6974    0.7567    0.7258      1837
     SER&ACC     0.7566    0.7559    0.7563      5420
     STORAGE     1.0000    0.0038    0.0077       260

   micro avg     0.7618    0.7822    0.7719     51838
   macro avg     0.7653    0.6934    0.6796     51838
weighted avg     0.7632    0.7822    0.7700     51838



## 7. Lưu mô hình

In [8]:
import os

os.makedirs(MODEL_SAVE, exist_ok=True)
model.save_pretrained(MODEL_SAVE)
tokenizer.save_pretrained(MODEL_SAVE)
print(f"Model saved -> {MODEL_SAVE}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved -> /content/models/NER
